In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("storageLocation","saprojectbankdeveastus")

In [0]:
%python
storageLocation = dbutils.widgets.get("storageLocation")

## EXTERNAL LOCATIONS (extl)

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `ext-raw`
URL  'abfss://raw@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ext_dev_metastore_root_azuremanagedidentity_1784080741220` )
COMMENT 'Ubicacion externa para las tablas de catalog del data lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `ext-bronze`
URL 'abfss://bronze@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ext_dev_metastore_root_azuremanagedidentity_1784080741220` )
COMMENT 'Unicacion para las tablas de bronze';


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `ext-silver`
URL 'abfss://silver@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ext_dev_metastore_root_azuremanagedidentity_1784080741220` )
COMMENT 'Unicacion para las tablas de silver';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `ext-gold`
URL 'abfss://gold@${storageLocation}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ext_dev_metastore_root_azuremanagedidentity_1784080741220` )
COMMENT 'Unicacion para las tablas de gold';

## Creacion del Catalog

In [0]:
%sql
-- 1. Crear Catalog (si no existe)
CREATE CATALOG IF NOT EXISTS bank_dev;


##Creacion de los SCHEMA's

Creando schemas por capas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bank_dev.bronze
MANAGED LOCATION 'abfss://bronze@${storageLocation}.dfs.core.windows.net/administrado/';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bank_dev.silver
MANAGED LOCATION 'abfss://silver@${storageLocation}.dfs.core.windows.net/administrado/';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bank_dev.gold
MANAGED LOCATION 'abfss://gold@${storageLocation}.dfs.core.windows.net/administrado/';

In [0]:
%sql
USE CATALOG bank_dev;

In [0]:
%sql
-- 1. Transactions (la mas importante)
CREATE TABLE IF NOT EXISTS bank_dev.bronze.transactions (
    id                  BIGINT,
    date                TIMESTAMP,
    client_id           BIGINT,
    card_id             BIGINT,
    amount              STRING,      
    use_chip            STRING,
    merchant_id         BIGINT,
    merchant_city       STRING,
    merchant_state      STRING,
    zip                 DOUBLE,
    mcc                 INT,
    errors              STRING,
    ingestion_date      TIMESTAMP
)
USING DELTA
CLUSTER BY (client_id, mcc, date)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bank_dev.bronze.users (
    id                  BIGINT,
    current_age         INT,
    retirement_age      INT,
    birth_year          INT,
    birth_month         INT,
    gender              STRING,
    address             STRING,
    latitude            DOUBLE,
    longitude           DOUBLE,
    per_capita_income   STRING,
    yearly_income       STRING,
    total_debt          STRING,
    credit_score        INT,
    num_credit_cards    INT,
    ingestion_date      TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- 3. Cards
CREATE TABLE IF NOT EXISTS bank_dev.bronze.cards (
    id                    BIGINT,
    client_id             BIGINT,
    card_brand            STRING,
    card_type             STRING,
    card_number           STRING,
    expires               STRING,
    cvv                   STRING,
    has_chip              STRING,
    num_cards_issued      INT,
    credit_limit          STRING,
    acct_open_date        STRING,
    year_pin_last_changed INT,
    card_on_dark_web      STRING,
    ingestion_date        TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- 4. MCC Codes
CREATE TABLE IF NOT EXISTS bank_dev.bronze.mcc_codes (
    mcc                 STRING,
    description         STRING,
    ingestion_date      TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- 5. Fraud Labels (Train)
CREATE TABLE IF NOT EXISTS bank_dev.bronze.fraud_labels (
    transaction_id      BIGINT,     -- o el nombre de la key que tenga
    target              STRING,     -- "Yes" / "No"
    ingestion_date      TIMESTAMP
)
USING DELTA;